# Import libraries

In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain_community.document_transformers import LongContextReorder
from langchain_classic.schema import Document
from langchain_core.prompts import PromptTemplate
from operator import itemgetter

load_dotenv()

True

# Get existing vectorstore

In [2]:
api_key = os.getenv("API_KEY")
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

In [ ]:
collection_name = "langchain_docs_index"
vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

# Initialize retriever

## MMR retriever

In [8]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    k=4,  # number of documents to retrieve after mmr
    fetch_k=20,  # number of documents to fetch in the first step
    # Lambda mult is a number between 0 and 1 that determines the degree
    # of diversity among the results with 0 corresponding to maximum diversity
    # and 1 to minimum diversity.
    lambda_mult=0.5,
)

In [9]:
mmr_retriever.search_type

'mmr'

In [10]:
relevant_docs = mmr_retriever.invoke("What is BPE?")
relevant_docs

[Document(id='55a6a0a4-4c49-5d5e-bd2b-e6dd375a906c', metadata={'contains_code_or_cli': False, 'talks_about_tokenization': True, 'talks_about_ngrams': False, 'page': 12, 'talks_about_morphology': False, 'talks_about_language_modeling': False, 'content_type': 'Narrative', 'volume': 1, 'contains_regex': False, 'linguistic_focus': '', 'source': 'data\\raw\\book_chapter_02.pdf', 'total_pages': 34, 'importance_score': 'High', 'contains_table': False, 'contains_math_latex': False, 'creationdate': 'D:20260329110007', 'chapter': 2}, page_content='2.4.3\nBPE in practice\nThe example above just showed simple BPE learning from sequences of ASCII\nbytes. How does BPE work with Unicode input? We normally run BPE on the\nindividual bytes of UTF-8-encoded text. That is, we take a Unicode representations\nof text as a series of code points, encode it in bytes using UTF-8, and we treat each of\nthese individual bytes as the input to BPE. Thus BPE likely begins by rediscovering\nthe 2-byte and common 3-b

# Lost in the middle reranking

In [11]:
reordering = LongContextReorder()
reordered_docs = list(reordering.transform_documents(relevant_docs))
reordered_docs

[Document(id='0fc344b3-3966-5cc2-888c-fc3cdee8324e', metadata={'talks_about_language_modeling': True, 'talks_about_ngrams': False, 'talks_about_morphology': False, 'chapter': 2, 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_math_latex': False, 'contains_table': False, 'page': 22, 'contains_code_or_cli': False, 'total_pages': 34, 'contains_regex': True, 'content_type': 'Narrative', 'linguistic_focus': '', 'importance_score': 'High', 'volume': 1, 'talks_about_tokenization': True, 'creationdate': 'D:20260329110007'}, page_content='BPE tokenization algorithm builds up tokens from sequences of characters inside\nwords and doesn’t tokenize across word boundaries.\nHere’s the regular expression used to do this pretokenization that is used for one\ninﬂuential language model, the GPT-2 language model (Radford et al., 2019):'),
 Document(id='ab9aa742-4738-5877-bcb5-380bbc8045db', metadata={'talks_about_morphology': False, 'importance_score': 'Medium', 'source': 'data\\raw\\book_chapter_0

## Use it in a RAG pipeline

In [12]:
def combine_documents(documents: list[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in documents])


prompt = PromptTemplate.from_template(
    """Given the following text extracts:
-----
{context}
-----
                                      
Answer the following question, if you don't know the answer, just write "I don't know."

Question: {question}"""
)

api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=1500,
    timeout=None,
    max_retries=2
)

In [13]:
stuff_chain = (
    {
        "context": itemgetter("question")
        | mmr_retriever
        | reordering.transform_documents
        | combine_documents,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
)

In [17]:
response = stuff_chain.invoke(input={"question": "What is BPE?"})

In [18]:
print(response)

BPE stands for BYTE-PAIR ENCODING. It is a tokenization algorithm that builds up tokens from sequences of characters inside words and doesn't tokenize across word boundaries.
